# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




Fix added before the demo: the current quay.io/jupyter/pyspark-notebook image ships Spark 4.2.0, and with the connector spark-sql-kafka 4.1.0 of the next cell every streaming query fails with a NoSuchMethodError (HDFSMetadataLog). I create the session here with the connector of the same version as Spark, the next cell then just gets this session back with getOrCreate.

In [2]:
import pyspark

spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}") \
    .getOrCreate()
print(spark.version)

4.2.0


In [3]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0") \
    .getOrCreate()


In [4]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [5]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [6]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [7]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

## Lab work

The demo above reads the topic wikistreams, filled by wikistream_producer.py: French Wikipedia edits replayed from the history (since 20260209). In the first part I add other aggregations on this topic, with tumbling and overlapping windows. In the second part I use my own producer, wikistream_producer_custom.py, which follows the live edits of several wikis on a second topic, wikistreams_live.

Order to run things on the host: admin_custom.py (creates both topics), then wikistream_producer.py and wikistream_producer_custom.py in two terminals, then this notebook.

In [8]:
import time
from pyspark.sql import functions as F
from pyspark.sql.functions import timestamp_seconds

# The data is small: 4 shuffle partitions instead of 200 keeps every micro-batch fast
spark.conf.set("spark.sql.shuffle.partitions", "4")
# Show the times in UTC, like the Wikimedia events (meta.dt)
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Same fields as the demo, plus the wiki, the namespace and the source Kafka topic of Wikimedia
wiki_schema = StructType(simple_wikimedia_schema.fields + [
    StructField("server_name", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("meta", StructType([
        StructField("dt", StringType(), True),
        StructField("topic", StringType(), True),
    ]), True),
])


def parse_wiki_events(raw):
    """Parse the Kafka messages and add the columns used by the aggregations."""
    return (raw
        .select(col("timestamp").alias("kafka_time"),
                from_json(col("value").cast("string"), wiki_schema).alias("data"))
        .select("kafka_time", "data.*")
        # A real timestamp column (from_unixtime returns a string), needed by withWatermark
        .withColumn("event_time", timestamp_seconds("timestamp"))
        .withColumn("size_delta", F.coalesce(col("length.new"), F.lit(0)) - F.coalesce(col("length.old"), F.lit(0)))
        .withColumn("editor", F.when(col("bot"), "bot").otherwise("human")))


def start_memory_query(df, name, mode):
    """Run a streaming query into an in-memory table that spark.sql can read."""
    return (df.writeStream
        .format("memory")
        .queryName(name)
        .outputMode(mode)
        .trigger(processingTime="5 seconds")
        .start())


def wait_for(query, max_seconds=90):
    """Wait until a micro-batch finds no new data in Kafka, or at most max_seconds."""
    deadline = time.time() + max_seconds
    while time.time() < deadline:
        progress = query.lastProgress
        if progress and progress["numInputRows"] == 0 and not query.status["isDataAvailable"]:
            return
        time.sleep(2)


events_df = parse_wiki_events(raw_df)

### Exploration 1: tumbling windows of 15 minutes

Each edit belongs to exactly one 15 minute window. For humans and bots separately I count the edits and the pages, the bytes added and removed, the average size of an edit and the share of minor edits.

In [9]:
tumbling_df = (events_df
    .groupBy(window("event_time", "15 minutes"), "editor")
    .agg(F.count("*").alias("edits"),
         F.approx_count_distinct("title").alias("pages"),
         F.sum(F.greatest("size_delta", F.lit(0))).alias("bytes_added"),
         F.sum(F.least("size_delta", F.lit(0))).alias("bytes_removed"),
         F.round(F.avg("size_delta"), 1).alias("avg_delta"),
         F.round(F.avg(col("minor").cast("int")) * 100, 1).alias("minor_pct"))
    .select(col("window.start").alias("window_start"), col("window.end").alias("window_end"),
            "editor", "edits", "pages", "bytes_added", "bytes_removed", "avg_delta", "minor_pct"))

tumbling_query = start_memory_query(tumbling_df, "tumbling_15min", "complete")
wait_for(tumbling_query)
spark.sql("SELECT * FROM tumbling_15min ORDER BY window_start, editor").show(16, truncate=False)

+-------------------+-------------------+------+-----+-----+-----------+-------------+---------+---------+
|window_start       |window_end         |editor|edits|pages|bytes_added|bytes_removed|avg_delta|minor_pct|
+-------------------+-------------------+------+-----+-----+-----------+-------------+---------+---------+
|2026-09-23 09:30:00|2026-09-23 09:45:00|bot   |2    |2    |728        |0            |364.0    |0.0      |
|2026-09-23 09:30:00|2026-09-23 09:45:00|human |16   |13   |531        |-216         |19.7     |50.0     |
|2026-09-23 09:45:00|2026-09-23 10:00:00|bot   |55   |56   |7290       |-217         |128.6    |3.6      |
|2026-09-23 09:45:00|2026-09-23 10:00:00|human |224  |164  |55795      |-19955       |160.0    |33.9     |
|2026-09-23 10:00:00|2026-09-23 10:15:00|bot   |60   |60   |8817       |-14792       |-99.6    |3.3      |
|2026-09-23 10:00:00|2026-09-23 10:15:00|human |215  |168  |72203      |-22551       |230.9    |23.3     |
|2026-09-23 10:15:00|2026-09-23 10:30

In [10]:
# Totals per editor type over all the windows
spark.sql("""
    SELECT editor, COUNT(*) AS windows, SUM(edits) AS edits,
           SUM(bytes_added) AS bytes_added, SUM(bytes_removed) AS bytes_removed,
           ROUND(SUM(bytes_added + bytes_removed) / SUM(edits), 1) AS avg_delta,
           ROUND(AVG(edits), 1) AS avg_edits_per_window, MAX(edits) AS max_edits_per_window
    FROM tumbling_15min GROUP BY editor
""").show()

+------+-------+-----+-----------+-------------+---------+--------------------+--------------------+
|editor|windows|edits|bytes_added|bytes_removed|avg_delta|avg_edits_per_window|max_edits_per_window|
+------+-------+-----+-----------+-------------+---------+--------------------+--------------------+
|   bot|     41| 1854|     386193|      -103765|    152.3|                45.2|                 112|
| human|     41| 7619|    2737359|      -543206|    288.0|               185.8|                 363|
+------+-------+-----+-----------+-------------+---------+--------------------+--------------------+



Over the 41 windows of 15 minutes, humans made 7,619 edits and bots 1,854, so bots are about 20% of the activity on French Wikipedia in this replay. Humans add more content per edit (288 bytes on average against 152 for bots) and they mark around a quarter of their edits as minor, while bots almost never do. The activity is quite regular, around 186 human edits per window with a peak at 363. The negative average for bots at 10:00 (-99.6) probably comes from one ListeriaBot list update that removed 13,766 bytes.

### Exploration 2: overlapping (sliding) windows of 1 hour, every 15 minutes

Windows are 1 hour long and a new one starts every 15 minutes, so they overlap and each edit is counted in 4 windows. This gives a smoother hourly activity curve than the tumbling windows.

In [11]:
sliding_df = (events_df
    .groupBy(window("event_time", "1 hour", "15 minutes"))
    .agg(F.count("*").alias("edits"),
         F.sum(col("bot").cast("int")).alias("bot_edits"),
         F.approx_count_distinct("user").alias("editors"),
         F.sum(F.abs("size_delta")).alias("bytes_changed"))
    .withColumn("bot_pct", F.round(col("bot_edits") / col("edits") * 100, 1))
    .select(col("window.start").alias("window_start"), col("window.end").alias("window_end"),
            "edits", "bot_edits", "bot_pct", "editors", "bytes_changed"))

sliding_query = start_memory_query(sliding_df, "sliding_1h", "complete")
wait_for(sliding_query)
spark.sql("SELECT * FROM sliding_1h ORDER BY window_start").show(12, truncate=False)

+-------------------+-------------------+-----+---------+-------+-------+-------------+
|window_start       |window_end         |edits|bot_edits|bot_pct|editors|bytes_changed|
+-------------------+-------------------+-----+---------+-------+-------+-------------+
|2026-09-23 08:45:00|2026-09-23 09:45:00|18   |2        |11.1   |14     |1475         |
|2026-09-23 09:00:00|2026-09-23 10:00:00|297  |57       |19.2   |124    |84732        |
|2026-09-23 09:15:00|2026-09-23 10:15:00|572  |117      |20.5   |198    |203095       |
|2026-09-23 09:30:00|2026-09-23 10:30:00|777  |161      |20.7   |231    |304705       |
|2026-09-23 09:45:00|2026-09-23 10:45:00|1007 |188      |18.7   |289    |381188       |
|2026-09-23 10:00:00|2026-09-23 11:00:00|937  |169      |18.0   |254    |401619       |
|2026-09-23 10:15:00|2026-09-23 11:15:00|932  |156      |16.7   |247    |368483       |
|2026-09-23 10:30:00|2026-09-23 11:30:00|967  |159      |16.4   |267    |342960       |
|2026-09-23 10:45:00|2026-09-23 

In [12]:
# With overlapping windows the same edit is counted several times
spark.sql("""
    SELECT 'tumbling 15 min' AS windows, COUNT(DISTINCT window_start) AS n_windows, SUM(edits) AS counted_edits FROM tumbling_15min
    UNION ALL
    SELECT 'sliding 1 h / 15 min', COUNT(*), SUM(edits) FROM sliding_1h
""").show(truncate=False)

+--------------------+---------+-------------+
|windows             |n_windows|counted_edits|
+--------------------+---------+-------------+
|tumbling 15 min     |41       |9473         |
|sliding 1 h / 15 min|47       |37892        |
+--------------------+---------+-------------+



The sliding windows give 47 windows instead of 41 and count 37,892 edits, exactly 4 times the 9,473 edits of the topic, because each edit falls in 4 overlapping windows. So I can't add up the sliding windows, but each one gives a real hourly figure that moves every 15 minutes, which makes a smoother curve. On September 23 the activity climbs from 18 to about 1,000 edits per hour between 9:45 and 10:45 UTC and then stays between 930 and 1,050, with 16 to 21% of bot edits. I think the low first windows only come from the replay starting at 9:44.

### Exploration 3: size of the edits and most active users

The first query has no aggregation: it only filters the stream, so it can run in append mode and each big edit (10,000 bytes or more added or removed) comes out once. The second one is a running count per user, in complete mode.

In [13]:
big_edits_df = (events_df
    .filter(F.abs("size_delta") >= 10000)
    .select("event_time", "title", "user", "editor", "size_delta", "comment"))

big_edits_query = start_memory_query(big_edits_df, "big_edits", "append")

users_df = (events_df
    .groupBy("user", "editor")
    .agg(F.count("*").alias("edits"),
         F.approx_count_distinct("title").alias("pages"),
         F.sum("size_delta").alias("net_bytes")))

users_query = start_memory_query(users_df, "top_users", "complete")
wait_for(big_edits_query)
wait_for(users_query)

spark.sql("SELECT * FROM big_edits ORDER BY abs(size_delta) DESC").show(10, truncate=60)
spark.sql("SELECT * FROM top_users ORDER BY edits DESC").show(10, truncate=False)

+-------------------+------------------------------------------------------------+--------------+------+----------+------------------------------------------------------------+
|         event_time|                                                       title|          user|editor|size_delta|                                                     comment|
+-------------------+------------------------------------------------------------+--------------+------+----------+------------------------------------------------------------+
|2026-09-30 03:46:09|                                 Âge d'or du cinéma argentin|         Bleff| human|    262881|Réécriture complète à partir de la traduction de [[:en:Go...|
|2026-09-23 14:41:38|                                                     Burkini|       Eskivor| human|    -47028|                déplacement infos vers [[Burkini en France]]|
|2026-09-23 13:22:07|                                               Lobatus gigas|         FredD| human|    -33166|

The biggest edit is a full rewrite of the article Âge d'or du cinéma argentin (+262,881 bytes, translated from English), and the biggest removals are content moved to another article or pages turned into redirects. OrlodrimBot and CodexBot are the most active accounts with 769 and 678 edits, and the most active human, Gnace2835, added 509,073 bytes in 236 edits. CodexBot also shows up with bot = false for 95 edits, so I guess the bot flag depends on the edit and not only on the account. The big edits query keeps no state at all, which is why append mode works there.

### Exploration 4: event time vs processing time

kafka_time is when the producer sent the message, close to the processing time, while event_time is when the edit happened on Wikipedia. Wikimedia replays the history from two datacenters (codfw and eqiad), so I group by meta.topic.

In [14]:
lag_df = (events_df
    .groupBy(col("meta.topic").alias("wikimedia_topic"))
    .agg(F.count("*").alias("edits"),
         F.min("event_time").alias("first_event"),
         F.max("event_time").alias("last_event"),
         F.round(F.avg(F.unix_timestamp("kafka_time") - col("timestamp")) / 3600, 1).alias("avg_lag_hours")))

lag_query = start_memory_query(lag_df, "event_lag", "complete")
wait_for(lag_query)
spark.sql("SELECT * FROM event_lag ORDER BY first_event").show(truncate=False)

+----------------------------+-----+-------------------+-------------------+-------------+
|wikimedia_topic             |edits|first_event        |last_event         |avg_lag_hours|
+----------------------------+-----+-------------------+-------------------+-------------+
|codfw.mediawiki.recentchange|8491 |2026-09-23 09:44:09|2026-09-23 17:03:36|331.6        |
|eqiad.mediawiki.recentchange|982  |2026-09-30 02:38:52|2026-09-30 04:49:04|173.5        |
+----------------------------+-----+-------------------+-------------------+-------------+



The 9,473 events come from two Wikimedia datacenters that replay different periods: codfw from September 23 (9:44 to 17:03 UTC, 8,491 edits) and eqiad from September 30 (982 edits). The lag between an edit and its arrival in Kafka is about 332 hours (almost 14 days) for the first one and 174 hours (7 days) for the second. So with since, the processing time says nothing about when the edits happened, and only event_time makes sense to build the windows.

### Exploration 5: watermark on the replayed data

Same tumbling windows with a watermark of 1 hour, in append mode: a window is written once, when the watermark passes its end, and older events are dropped. maxOffsetsPerTrigger cuts the topic into micro-batches of 500 messages, like a real stream would arrive (in one single batch the watermark would only move at the end).

In [15]:
replay_batches_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "wikistreams")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 500)
    .load())

watermarked_df = (parse_wiki_events(replay_batches_df)
    .withWatermark("event_time", "1 hour")
    .groupBy(window("event_time", "15 minutes"))
    .agg(F.count("*").alias("edits"))
    .select(col("window.start").alias("window_start"), col("window.end").alias("window_end"), "edits"))

watermark_query = start_memory_query(watermarked_df, "watermarked_15min", "append")
wait_for(watermark_query, max_seconds=300)

# How the watermark moved, batch by batch (numRowsDroppedByWatermark counts partial aggregates, not events)
print("micro-batches run:", len(watermark_query.recentProgress))
for p in watermark_query.recentProgress[:6]:
    dropped = sum(s["numRowsDroppedByWatermark"] for s in p["stateOperators"])
    print(p["batchId"], p["numInputRows"], "rows, max event", p["eventTime"].get("max"),
          "watermark", p["eventTime"].get("watermark"), "dropped", dropped)

# Windows written with the watermark vs the same windows without it
spark.sql("""
    SELECT t.window_start, t.edits AS edits_no_watermark, w.edits AS edits_watermark_1h
    FROM (SELECT window_start, SUM(edits) AS edits FROM tumbling_15min GROUP BY window_start) t
    LEFT JOIN watermarked_15min w ON t.window_start = w.window_start
    ORDER BY t.window_start
""").show(40, truncate=False)

micro-batches run: 20
0 500 rows, max event 2026-09-30T02:46:05.000Z watermark 1970-01-01T00:00:00.000Z dropped 0
1 500 rows, max event 2026-09-30T02:53:32.000Z watermark 2026-09-30T01:46:05.000Z dropped 0
2 500 rows, max event 2026-09-30T02:58:32.000Z watermark 2026-09-30T01:53:32.000Z dropped 3
3 500 rows, max event 2026-09-30T03:03:37.000Z watermark 2026-09-30T01:58:32.000Z dropped 3
4 500 rows, max event 2026-09-30T03:12:05.000Z watermark 2026-09-30T02:03:37.000Z dropped 2
5 500 rows, max event 2026-09-30T03:17:27.000Z watermark 2026-09-30T02:12:05.000Z dropped 3
+-------------------+------------------+------------------+
|window_start       |edits_no_watermark|edits_watermark_1h|
+-------------------+------------------+------------------+
|2026-09-23 09:30:00|18                |18                |
|2026-09-23 09:45:00|279               |279               |
|2026-09-23 10:00:00|275               |275               |
|2026-09-23 10:15:00|205               |205               |
|2026-

The watermark jumps to September 30 right after the first batch, because the producer mixes the two datacenters and the first 500 messages already contain edits of the 30th. From then on almost every September 23 edit is too late: only the windows that were already in the state before the jump are written, and in the end 1,316 of the 9,473 edits are in written windows (847 of the 23rd and 469 of the 30th). The windows of the 30th after 03:45 are not lost, they are just waiting for the watermark to pass their end. numRowsDroppedByWatermark stays at 2 or 3 per batch because it counts partial aggregates per window, not events. I think it shows that a watermark only works if the event times arrive more or less in order, which is not the case with this replay.

### Filtering at the producer level

wikistream_producer_custom.py takes its filters as arguments and registers them with stream.register_filter: the wikis (server_name), the event types, specific page titles, namespaces, and bots can be excluded. With no since argument it reads the live stream instead of the history. Examples:

python wikistream_producer_custom.py (live edits and page creations of fr, en and de Wikipedia, topic wikistreams_live, 10 minutes)

python wikistream_producer_custom.py --wikis en.wikipedia.org --titles "Taylor Swift" "Donald Trump" --topic wikistreams_pages

python wikistream_producer_custom.py --wikis www.wikidata.org --humans-only --namespaces 0

The filters of register_filter are applied in the client: pywikibot still receives the whole Wikimedia stream and drops what doesn't match, so filtering at the producer level mostly saves Kafka and Spark, not the network. I ran the first example for 10 minutes and it sent 1,311 live edits and page creations from the three Wikipedias to wikistreams_live. Without since the events arrive almost right after the edits, so the event time and the processing time are close, which is what Explorations 6 and 7 need.

### Exploration 6: live stream of several wikis, tumbling windows of 1 minute with a watermark

This part reads wikistreams_live. The data is live, so the event time is close to the processing time and a short watermark (2 minutes) is enough. In append mode a 1 minute window only shows up once the watermark has passed its end.

In [16]:
live_raw_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:19092")
    .option("subscribe", "wikistreams_live")
    .option("startingOffsets", "earliest")
    .load())

live_df = parse_wiki_events(live_raw_df).withWatermark("event_time", "2 minutes")

live_tumbling_df = (live_df
    .groupBy(window("event_time", "1 minute"), "server_name")
    .agg(F.count("*").alias("edits"),
         F.sum(col("bot").cast("int")).alias("bot_edits"),
         F.sum(F.abs("size_delta")).alias("bytes_changed"))
    .select(col("window.start").alias("window_start"), "server_name", "edits", "bot_edits", "bytes_changed"))

live_tumbling_query = start_memory_query(live_tumbling_df, "live_tumbling_1min", "append")

# Same aggregation printed in the Docker logs (docker logs -f pyspark_notebook), in update mode
live_console_query = (live_tumbling_df.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .start())

wait_for(live_tumbling_query, max_seconds=120)
spark.sql("SELECT * FROM live_tumbling_1min ORDER BY window_start DESC, server_name").show(15, truncate=False)

+-------------------+----------------+-----+---------+-------------+
|window_start       |server_name     |edits|bot_edits|bytes_changed|
+-------------------+----------------+-----+---------+-------------+
|2026-10-07 09:13:00|de.wikipedia.org|24   |1        |3078         |
|2026-10-07 09:13:00|en.wikipedia.org|98   |17       |39805        |
|2026-10-07 09:13:00|fr.wikipedia.org|20   |6        |9946         |
|2026-10-07 09:12:00|de.wikipedia.org|24   |1        |2921         |
|2026-10-07 09:12:00|en.wikipedia.org|84   |20       |21667        |
|2026-10-07 09:12:00|fr.wikipedia.org|21   |6        |2317         |
|2026-10-07 09:11:00|de.wikipedia.org|27   |2        |3060         |
|2026-10-07 09:11:00|en.wikipedia.org|86   |25       |20704        |
|2026-10-07 09:11:00|fr.wikipedia.org|22   |1        |8966         |
|2026-10-07 09:10:00|de.wikipedia.org|23   |3        |49140        |
|2026-10-07 09:10:00|en.wikipedia.org|93   |24       |32725        |
|2026-10-07 09:10:00|fr.wikipedia.

With live data each minute gets a closed window per wiki: English Wikipedia has about 80 to 100 edits per minute, German around 25 and French 20 to 26 (with one quieter minute of only 9 edits). A window only appears in the append table once the watermark (latest event minus 2 minutes) has passed its end, so the last two minutes of the run are never written. In the Docker logs the console query in update mode shows the same windows growing batch after batch instead.

### Exploration 7: live stream, overlapping windows of 5 minutes every minute

Rolling 5 minute activity per wiki, updated every minute. Each edit lands in 5 windows.

In [17]:
live_sliding_df = (live_df
    .groupBy(window("event_time", "5 minutes", "1 minute"), "server_name")
    .agg(F.count("*").alias("edits"),
         F.approx_count_distinct("user").alias("editors"),
         F.round(F.avg(col("bot").cast("int")) * 100, 1).alias("bot_pct"),
         F.round(F.avg(F.abs("size_delta")), 0).alias("avg_abs_delta"))
    .select(col("window.start").alias("window_start"), col("window.end").alias("window_end"),
            "server_name", "edits", "editors", "bot_pct", "avg_abs_delta"))

live_sliding_query = start_memory_query(live_sliding_df, "live_sliding_5min", "append")
wait_for(live_sliding_query, max_seconds=120)
spark.sql("SELECT * FROM live_sliding_5min ORDER BY window_start DESC, server_name").show(15, truncate=False)

+-------------------+-------------------+----------------+-----+-------+-------+-------------+
|window_start       |window_end         |server_name     |edits|editors|bot_pct|avg_abs_delta|
+-------------------+-------------------+----------------+-----+-------+-------+-------------+
|2026-10-07 09:09:00|2026-10-07 09:14:00|de.wikipedia.org|123  |57     |7.3    |523.0        |
|2026-10-07 09:09:00|2026-10-07 09:14:00|en.wikipedia.org|443  |211    |25.3   |318.0        |
|2026-10-07 09:09:00|2026-10-07 09:14:00|fr.wikipedia.org|98   |62     |17.3   |270.0        |
|2026-10-07 09:08:00|2026-10-07 09:13:00|de.wikipedia.org|118  |62     |8.5    |540.0        |
|2026-10-07 09:08:00|2026-10-07 09:13:00|en.wikipedia.org|431  |207    |26.9   |317.0        |
|2026-10-07 09:08:00|2026-10-07 09:13:00|fr.wikipedia.org|97   |62     |12.4   |234.0        |
|2026-10-07 09:07:00|2026-10-07 09:12:00|de.wikipedia.org|109  |64     |11.0   |661.0        |
|2026-10-07 09:07:00|2026-10-07 09:12:00|en.wikipe

In [18]:
# Average per wiki over all the closed 5 minute windows
spark.sql("""
    SELECT server_name, COUNT(*) AS windows, ROUND(AVG(edits), 1) AS avg_edits_per_5min,
           ROUND(AVG(bot_pct), 1) AS avg_bot_pct, ROUND(AVG(avg_abs_delta), 0) AS avg_abs_delta
    FROM live_sliding_5min GROUP BY server_name ORDER BY avg_edits_per_5min DESC
""").show(truncate=False)

+----------------+-------+------------------+-----------+-------------+
|server_name     |windows|avg_edits_per_5min|avg_bot_pct|avg_abs_delta|
+----------------+-------+------------------+-----------+-------------+
|en.wikipedia.org|9      |286.8             |25.8       |255.0        |
|de.wikipedia.org|8      |78.8              |11.3       |458.0        |
|fr.wikipedia.org|8      |73.4              |14.5       |184.0        |
+----------------+-------+------------------+-----------+-------------+



Over the closed 5 minute windows, English Wikipedia has about 287 edits, German 79 and French 73, but the averages are probably a bit low because the first windows started before the producer did. Bots make 26% of the English edits, against 15% on French and 11% on German Wikipedia. German edits are the biggest on average this time (458 bytes against 255 for English and 184 for French), probably because of a few large edits like the 49,140 bytes changed in the 9:10 minute. Each edit is counted in 5 windows, so these figures can be read as a rolling 5 minute activity refreshed every minute.

In [19]:
# Active queries; uncomment the loop to stop them all when done
for q in spark.streams.active:
    print(q.name, q.status["message"])

#for q in spark.streams.active:
#    q.stop()

top_users Waiting for next trigger
sliding_1h Waiting for next trigger
watermarked_15min Waiting for next trigger
big_edits Waiting for next trigger
None Waiting for data to arrive
live_tumbling_1min Waiting for next trigger
tumbling_15min Waiting for next trigger
live_sliding_5min Waiting for next trigger
event_lag Waiting for next trigger
None Waiting for data to arrive
